# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Best combined model: OOF analysis and test-group audit
# ------------------------------
# Best verified model: TF-IDF + prompt-relative response patterns.
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score

BEST_CHAR_RANGE = (1, 5)
BEST_C = 2.0
OOF_OUTPUT = Path('/kaggle/working/combined_oof_results.json')

FEATURE_NAMES = [
    'characters', 'words', 'sentences', 'punctuation', 'digits', 'uppercase',
    'question_marks', 'exclamation_marks', 'colons', 'semicolons',
    'brackets', 'markdown_markers', 'bullet_markers', 'newline_count',
]

def response_pattern(text):
    words = re.findall(r"\b[\w']+\b", text)
    return [
        float(len(text)), float(len(words)),
        float(len(re.findall(r'[.!?]+', text))),
        float(sum(text.count(c) for c in PUNCTUATION)),
        float(sum(c.isdigit() for c in text)),
        float(sum(c.isupper() for c in text)),
        float(text.count('?')), float(text.count('!')),
        float(text.count(':')), float(text.count(';')),
        float(sum(text.count(c) for c in '()[]{}')),
        float(sum(text.count(c) for c in '#*_`')),
        float(sum(text.count(c) for c in '*-•')),
        float(text.count('\\n')),
    ]

absolute = np.asarray([response_pattern(text) for text in texts], dtype=np.float32)
relative = np.zeros_like(absolute)
for group_id in np.unique(groups):
    mask = groups == group_id
    values = absolute[mask]
    mean = values.mean(axis=0)
    std = values.std(axis=0)
    std[std < 1e-6] = 1.0
    relative[mask] = (values - mean) / std

folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
oof_predictions = np.empty(labels.shape, dtype=object)
fold_records = []
started = time.time()

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    word_vectorizer = make_word_vectorizer()
    X_word_train = word_vectorizer.fit_transform(texts[train_idx])
    X_word_valid = word_vectorizer.transform(texts[valid_idx])

    punctuation_scaler = StandardScaler()
    X_punct_train = csr_matrix(punctuation_scaler.fit_transform(punctuation[train_idx]))
    X_punct_valid = csr_matrix(punctuation_scaler.transform(punctuation[valid_idx]))

    char_vectorizer = make_char_vectorizer(BEST_CHAR_RANGE)
    X_char_train = char_vectorizer.fit_transform(texts[train_idx])
    X_char_valid = char_vectorizer.transform(texts[valid_idx])

    text_train = hstack([X_word_train, X_char_train, X_punct_train], format='csr')
    text_valid = hstack([X_word_valid, X_char_valid, X_punct_valid], format='csr')

    pattern_scaler = StandardScaler()
    pattern_train = csr_matrix(pattern_scaler.fit_transform(relative[train_idx]))
    pattern_valid = csr_matrix(pattern_scaler.transform(relative[valid_idx]))

    X_train = hstack([text_train, pattern_train], format='csr')
    X_valid = hstack([text_valid, pattern_valid], format='csr')

    model = LinearSVC(C=BEST_C, class_weight='balanced', max_iter=2_000)
    model.fit(X_train, labels[train_idx])
    predictions = model.predict(X_valid)
    oof_predictions[valid_idx] = predictions
    score = float(balanced_accuracy_score(labels[valid_idx], predictions))
    fold_records.append({'fold': fold_number, 'balanced_accuracy': score, 'validation_rows': int(len(valid_idx))})
    print(f'Fold {fold_number}/{N_SPLITS}: score={score:.4f}')

assert not pd.isna(oof_predictions).any()
classes = np.unique(labels)
score = float(balanced_accuracy_score(labels, oof_predictions))
cm = confusion_matrix(labels, oof_predictions, labels=classes)
report = classification_report(labels, oof_predictions, labels=classes, output_dict=True, zero_division=0)
per_class = []
for cls in classes:
    row = report[str(cls)]
    per_class.append({'model': cls, 'precision': float(row['precision']), 'recall': float(row['recall']), 'f1_score': float(row['f1-score']), 'support': int(row['support'])})
per_class.sort(key=lambda row: row['recall'])

# Verify whether a test file exposes complete prompt groups.
test_audit = {'test_file_found': False}
for candidate in [DATA_DIR / 'test.csv', DATA_DIR / 'test.parquet']:
    if candidate.exists():
        test_audit['test_file_found'] = True
        test_audit['test_file'] = str(candidate.name)
        if candidate.suffix == '.csv':
            test = pd.read_csv(candidate)
        else:
            test = pd.read_parquet(candidate)
        test_audit['rows'] = int(len(test))
        test_audit['columns'] = test.columns.tolist()
        if 'prompt_id' in test.columns:
            counts = test.groupby('prompt_id').size()
            test_audit['prompt_group_count'] = int(counts.size)
            test_audit['group_size_distribution'] = {str(k): int(v) for k,v in counts.value_counts().sort_index().items()}
            test_audit['all_groups_size_18'] = bool((counts == 18).all())
        break

summary = {
    'experiment': 'best TF-IDF plus prompt-relative OOF analysis',
    'split_method': '5-fold GroupKFold by prompt_id',
    'model_config': {'word_ngram_range': list(WORD_NGRAM_RANGE), 'char_ngram_range': list(BEST_CHAR_RANGE), 'linear_svc_C': BEST_C, 'prompt_features': FEATURE_NAMES},
    'oof_balanced_accuracy': score,
    'fold_records': fold_records,
    'per_class': per_class,
    'claude_fable_recall': next(row['recall'] for row in per_class if row['model'] == 'claude-fable-5'),
    'claude_fable_precision': next(row['precision'] for row in per_class if row['model'] == 'claude-fable-5'),
    'confusion_matrix': cm.tolist(),
    'classes': classes.tolist(),
    'test_group_audit': test_audit,
    'total_elapsed_seconds': round(time.time() - started, 2),
}
OOF_OUTPUT.write_text(json.dumps(summary, indent=2))
pd.DataFrame({'actual': labels, 'predicted': oof_predictions, 'prompt_id': groups}).to_csv('/kaggle/working/combined_oof_predictions.csv', index=False)
print(json.dumps({'oof_balanced_accuracy': score, 'claude_fable_recall': summary['claude_fable_recall'], 'claude_fable_precision': summary['claude_fable_precision'], 'test_group_audit': test_audit}, indent=2))


## Final verification

This cell verifies whether the prompt-relative feature can be constructed from the competition test layout and whether the combined model improves Claude-fable recall without sacrificing overall balanced accuracy.
